# End-to-End Apache Iceberg SQL Tutorial (Pure SQL)

This notebook walks through a complete implementation of an `orders` table utilizing Iceberg's advanced catalog procedures, Merge-on-Read mechanics, and the Write-Audit-Publish (WAP) pattern.

**First, run the initialization cell below.** It sets up a custom `%%sql` magic command so you can run native SQL directly in all the following cells.

In [11]:
from pyspark.sql import SparkSession
from IPython.core.magic import register_cell_magic

# Initialize Spark Session (Inherits configurations automatically from spark-defaults.conf)
spark = SparkSession.builder.appName('Iceberg-Pure-SQL-Tutorial').getOrCreate()

# Silence verbose Spark/Iceberg warnings
spark.sparkContext.setLogLevel("ERROR")

# Define %%sql magic for standard Jupyter environments
@register_cell_magic
def sql(line, cell):
    # 1. Remove all single-line SQL comments (anything from '--' to the end of a line)
    cell_without_comments = re.sub(r'--.*$', '', cell, flags=re.MULTILINE)
    
    # 2. Split by semicolon to allow multiple statements per cell
    queries = [q.strip() for q in cell_without_comments.split(';') if q.strip()]
    
    for query in queries:
        df = spark.sql(query)
        # Only display output if the query returns data (skips empty DDL/DML outputs)
        if len(df.columns) > 0:
            df.show(truncate=False)
            
print("Spark Session initialized. The %%sql magic is now ready to use!")

Spark Session initialized. The %%sql magic is now ready to use!


## 1. Scenario Setup & DDL

First, we isolate the domain by creating a dedicated namespace. 

When creating the table, we leverage **Hidden Partitioning**. Instead of forcing engineers to extract a physical `order_date` column, Iceberg's `days()` transform dynamically derives the partition from `order_timestamp`. 

We explicitly configure **Merge-on-Read (MoR)** for row-level operations to optimize write latency.

In [6]:
%%sql
CREATE NAMESPACE IF NOT EXISTS demo.ecommerce;

DROP TABLE IF EXISTS demo.ecommerce.orders;

CREATE TABLE demo.ecommerce.orders (
    order_id BIGINT,
    customer_id BIGINT,
    order_timestamp TIMESTAMP,
    status STRING,
    total_amount DECIMAL(10, 2)
)
USING iceberg
PARTITIONED BY (days(order_timestamp))
TBLPROPERTIES (
    'format-version' = '2',                        
    'write.update.mode' = 'merge-on-read',         
    'write.delete.mode' = 'merge-on-read',         
    'write.merge.mode' = 'merge-on-read',          
    'write.parquet.compression-codec' = 'zstd'     
);

In [7]:
%%sql

SELECT * FROM demo.ecommerce.orders.snapshots;             -- every snapshot + summary
SELECT * FROM demo.ecommerce.orders.history;                -- snapshot lineage / parent-child
SELECT * FROM demo.ecommerce.orders.manifests;               -- manifest list entries
SELECT * FROM demo.ecommerce.orders.files;                   -- every data file + stats
SELECT * FROM demo.ecommerce.orders.partitions;              -- partition-level stats
SELECT * FROM demo.ecommerce.orders.metadata_log_entries;    -- metadata.json version history
SELECT * FROM demo.ecommerce.orders.refs;                    -- branches & tags
SELECT * FROM demo.ecommerce.orders.all_data_files;          -- data files across all snapshots
SELECT * FROM demo.ecommerce.orders.all_manifests;           -- manifests across all snapshots


+------------+-----------+---------+---------+-------------+-------+
|committed_at|snapshot_id|parent_id|operation|manifest_list|summary|
+------------+-----------+---------+---------+-------------+-------+
+------------+-----------+---------+---------+-------------+-------+

+---------------+-----------+---------+-------------------+
|made_current_at|snapshot_id|parent_id|is_current_ancestor|
+---------------+-----------+---------+-------------------+
+---------------+-----------+---------+-------------------+

+-------+----+------+-----------------+-----------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+-------------------+
|content|path|length|partition_spec_id|added_snapshot_id|added_data_files_count|existing_data_files_count|deleted_data_files_count|added_delete_files_count|existing_delete_files_count|deleted_delete_files_count|partition_summaries|
+-------+----+----

In [8]:
metadata_file= "s3a://warehouse/ecommerce/orders/metadata/00000-dab58a5a-3302-4a76-8195-2890af74e71e.metadata.json"
from pyspark.sql.functions import explode,col

#md_json_df= spark.read.json(metadata_file)
md_json_df = spark.read.option("multiLine", "true").format("json").load(metadata_file)
#md_json_df.show()



# 1. Explode the array into rows
exploded_schemas_df = md_json_df.withColumn("schema_element", explode(col("schemas")))

# 2. Select the exploded column and flatten any structs inside it
schema_details = exploded_schemas_df.select("schema_element.*")

schema_details.show(truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------+
|fields                                                                                                                                                                     |schema-id|type  |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------+
|[{1, order_id, false, long}, {2, customer_id, false, long}, {3, order_timestamp, false, timestamptz}, {4, status, false, string}, {5, total_amount, false, decimal(10, 2)}]|0        |struct|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------+



In [ ]:
## Create a Customers Dimension Table with copy on write properties.

## 2. Data Ingestion & Core DML

We will populate the table with an initial batch of raw events, and then perform row-level mutations (UPDATE and DELETE).

In [9]:
%%sql
-- 1. Initial Ingestion
INSERT INTO demo.ecommerce.orders VALUES
    (1001, 5501, CAST('2026-09-10 08:15:00' AS TIMESTAMP), 'PENDING', 120.50),
    (1002, 5502, CAST('2026-09-11 09:30:00' AS TIMESTAMP), 'PENDING', 89.99),
    (1003, 5503, CAST('2026-09-12 14:45:00' AS TIMESTAMP), 'PROCESSING', 1045.00);





In [10]:
%%sql

SELECT * FROM demo.ecommerce.orders.snapshots;             -- every snapshot + summary
SELECT * FROM demo.ecommerce.orders.history;                -- snapshot lineage / parent-child
SELECT * FROM demo.ecommerce.orders.manifests;               -- manifest list entries
SELECT * FROM demo.ecommerce.orders.files;                   -- every data file + stats
SELECT * FROM demo.ecommerce.orders.partitions;              -- partition-level stats
SELECT * FROM demo.ecommerce.orders.metadata_log_entries;    -- metadata.json version history
SELECT * FROM demo.ecommerce.orders.refs;                    -- branches & tags
SELECT * FROM demo.ecommerce.orders.all_data_files;          -- data files across all snapshots
SELECT * FROM demo.ecommerce.orders.all_manifests;           -- manifests across all snapshots


+-----------------------+-------------------+---------+---------+--------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id|operation|manifest_list                                                                                                 |summary                                                                                                                                                                                      

In [13]:
manifest_file= "s3a://warehouse/ecommerce/orders/metadata/75be7f2c-8885-4a0f-951d-debc6f9e09e1-m0.avro"

manifest_df=spark.read.format("avro").load(manifest_file)

manifest_df.show(truncate=true)

AnalysisException: Failed to find data source: avro. Avro is built-in but external data source module since Spark 2.4. Please deploy the application as per the deployment section of Apache Avro Data Source Guide.

In [ ]:
%%sql

-- 2. Row-level Update

UPDATE demo.ecommerce.orders 
SET status = 'PROCESSING' 
WHERE order_id = 1001;

In [ ]:
%%sql

SELECT * FROM hive_catalog.nyc.taxis.snapshots;             -- every snapshot + summary
SELECT * FROM hive_catalog.nyc.taxis.history;                -- snapshot lineage / parent-child
SELECT * FROM hive_catalog.nyc.taxis.manifests;               -- manifest list entries
SELECT * FROM hive_catalog.nyc.taxis.files;                   -- every data file + stats
SELECT * FROM hive_catalog.nyc.taxis.partitions;              -- partition-level stats
SELECT * FROM hive_catalog.nyc.taxis.metadata_log_entries;    -- metadata.json version history
SELECT * FROM hive_catalog.nyc.taxis.refs;                    -- branches & tags
SELECT * FROM hive_catalog.nyc.taxis.all_data_files;          -- data files across all snapshots
SELECT * FROM hive_catalog.nyc.taxis.all_manifests;           -- manifests across all snapshots

SELECT * 
FROM demo.ecommerce.orders.history;

In [ ]:
%%sql
-- 3. Row-level Delete
DELETE FROM demo.ecommerce.orders 
WHERE order_id = 1002;


In [ ]:
%%sql

SELECT * 
FROM demo.ecommerce.orders.history;

Simulate an incremental batch arriving from a staging layer using a comprehensive `MERGE INTO` (Upsert) operation.

In [4]:
%%sql
WITH staging_updates AS (
    SELECT 1001 AS order_id, 5501 AS customer_id, CAST('2026-09-10 08:15:00' AS TIMESTAMP) AS order_timestamp, 'COMPLETED' AS status, 120.50 AS total_amount UNION ALL
    SELECT 1004 AS order_id, 5504 AS customer_id, CAST('2026-09-13 10:00:00' AS TIMESTAMP) AS order_timestamp, 'PENDING' AS status, 45.00 AS total_amount
)
MERGE INTO demo.ecommerce.orders t
USING staging_updates s
ON t.order_id = s.order_id
WHEN MATCHED AND s.status = 'COMPLETED' THEN 
    UPDATE SET t.status = s.status
WHEN NOT MATCHED THEN 
    INSERT (order_id, customer_id, order_timestamp, status, total_amount)
    VALUES (s.order_id, s.customer_id, s.order_timestamp, s.status, s.total_amount);

-- View the final state
SELECT * FROM demo.ecommerce.orders ORDER BY order_id;

+--------+-----------+-------------------+----------+------------+
|order_id|customer_id|order_timestamp    |status    |total_amount|
+--------+-----------+-------------------+----------+------------+
|1001    |5501       |2026-09-10 08:15:00|COMPLETED |120.50      |
|1003    |5503       |2026-09-12 14:45:00|PROCESSING|1045.00     |
|1004    |5504       |2026-09-13 10:00:00|PENDING   |45.00       |
+--------+-----------+-------------------+----------+------------+



## 3. Metadata Inspection (Under the Hood)

Iceberg exposes its internal architectural state via system tables. Querying these tables allows you to track exactly how the metadata tree evaluates the active dataset.

In [5]:
%%sql
-- Table History (Chronological log of snapshots)
SELECT * 
FROM demo.ecommerce.orders.history;

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-09-13 17:48:44.712|7790499068949806056|NULL               |true               |
|2026-09-13 17:48:45.429|6974023413306972610|7790499068949806056|true               |
|2026-09-13 17:48:45.588|3754598430136729173|6974023413306972610|true               |
|2026-09-13 17:48:46.274|3829320643997263216|3754598430136729173|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [6]:
%%sql
    
SELECT snapshot_id, operation, summary 
FROM demo.ecommerce.orders.snapshots;

+-------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |operation|summary                                                                                                                                                                                                                                                                                                                                                                                               

In [7]:
%%sql
-- Table History (Chronological log of snapshots)
SELECT made_current_at, snapshot_id, parent_id, is_current_ancestor 
FROM demo.ecommerce.orders.history;

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-09-13 17:48:44.712|7790499068949806056|NULL               |true               |
|2026-09-13 17:48:45.429|6974023413306972610|7790499068949806056|true               |
|2026-09-13 17:48:45.588|3754598430136729173|6974023413306972610|true               |
|2026-09-13 17:48:46.274|3829320643997263216|3754598430136729173|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [8]:
%%sql
-- Snapshots (Deep operational metrics like files added/removed)
SELECT snapshot_id, operation, summary 
FROM demo.ecommerce.orders.snapshots;

+-------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |operation|summary                                                                                                                                                                                                                                                                                                                                                                                               

In [9]:
%%sql
-- Files (Physical Parquet data and positional delete files)
SELECT file_path, file_format, record_count, file_size_in_bytes, partition 
FROM demo.ecommerce.orders.files;

+----------------------------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+------------+
|file_path                                                                                                                               |file_format|record_count|file_size_in_bytes|partition   |
+----------------------------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+------------+
|s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-09-13/00000-10-d13fdbdd-cee1-49ec-9fd2-f51a39b92f64-00001.parquet        |PARQUET    |1           |1682              |{2026-09-13}|
|s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-09-10/00000-10-d13fdbdd-cee1-49ec-9fd2-f51a39b92f64-00002.parquet        |PARQUET    |1           |1696              |{2026-09-10}|
|s3a://warehouse/eco

## 4. Time Travel & Versioning

Because Iceberg treats updates as immutable new snapshots, previous states of the table remain queryable. 
Below we query the table as it existed exactly 1 minute ago, before our latest updates.

In [10]:
%%sql
-- Time travel using a dynamic timestamp (1 minute ago)
SELECT * FROM demo.ecommerce.orders 
TIMESTAMP AS OF (current_timestamp() - INTERVAL 1 MINUTE);

-- Note: To permanently rollback the table to this state, you would use:
-- CALL demo.system.rollback_to_timestamp('ecommerce.orders', current_timestamp() - INTERVAL 1 MINUTE);

IllegalArgumentException: Cannot find a snapshot older than 2026-09-13T17:47:47.450+00:00

## 5. Branching & Tagging (Data Lifecycle)

The **Write-Audit-Publish (WAP)** pattern enables logical isolation. Engineers can write massive ETL jobs to a branch without downstream consumers seeing the changes until they are verified and fast-forwarded to main.

In [11]:
%%sql
-- 1. Create an isolated branch
ALTER TABLE demo.ecommerce.orders CREATE BRANCH IF NOT EXISTS etl_pipeline;

-- 2. Write explicitly to the branch (Main remains unaffected)
INSERT INTO demo.ecommerce.orders.branch_etl_pipeline
VALUES (1005, 5505, CAST('2026-09-13 11:30:00' AS TIMESTAMP), 'PENDING', 210.00);

-- 3. Query the branch to audit the new data
SELECT * FROM demo.ecommerce.orders VERSION AS OF 'etl_pipeline';

+--------+-----------+-------------------+----------+------------+
|order_id|customer_id|order_timestamp    |status    |total_amount|
+--------+-----------+-------------------+----------+------------+
|1003    |5503       |2026-09-12 14:45:00|PROCESSING|1045.00     |
|1005    |5505       |2026-09-13 11:30:00|PENDING   |210.00      |
|1004    |5504       |2026-09-13 10:00:00|PENDING   |45.00       |
|1001    |5501       |2026-09-10 08:15:00|COMPLETED |120.50      |
+--------+-----------+-------------------+----------+------------+



In [12]:
%%sql
-- 4. Fast-forward main branch to publish ETL changes instantly
CALL demo.system.fast_forward('ecommerce.orders', 'main', 'etl_pipeline');

-- 5. Create an audit tag for compliance (Retained for 1 year)
ALTER TABLE demo.ecommerce.orders CREATE TAG IF NOT EXISTS end_of_q3_2026 RETAIN 365 DAYS;

+--------------+-------------------+-------------------+
|branch_updated|previous_ref       |updated_ref        |
+--------------+-------------------+-------------------+
|main          |3829320643997263216|7442190996494522195|
+--------------+-------------------+-------------------+



## 6. Table Maintenance & Optimization

Streaming ingestion and MoR fragment the table into thousands of small Parquet files. Routine maintenance is mandatory.
*   **rewrite_data_files**: Consolidates small files into optimized target sizes.
*   **rewrite_manifests**: Aligns the metadata tree with the new data file structure.
*   **expire_snapshots**: Removes old snapshot references to manage storage costs.
*   **remove_orphan_files**: Safely sweeps physical files no longer tracked by metadata.

In [13]:
%%sql
-- 1. Run Binpack Compaction to merge small files (targeting 512MB per file)
CALL demo.system.rewrite_data_files(
  table => 'ecommerce.orders', 
  strategy => 'binpack', 
  options => map('target-file-size-bytes', '536870912')
);

-- 2. Rewrite Manifests to optimize query planning speed
CALL demo.system.rewrite_manifests('ecommerce.orders');

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|3                         |1                     |5041                 |0                      |
+--------------------------+----------------------+---------------------+-----------------------+

+-------------------------+---------------------+
|rewritten_manifests_count|added_manifests_count|
+-------------------------+---------------------+
|7                        |2                    |
+-------------------------+---------------------+



In [14]:
%%sql
-- Files (Physical Parquet data and positional delete files)
SELECT file_path, file_format, record_count, file_size_in_bytes, partition 
FROM demo.ecommerce.orders.files;

+----------------------------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+------------+
|file_path                                                                                                                               |file_format|record_count|file_size_in_bytes|partition   |
+----------------------------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+------------+
|s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-09-10/00000-23-9893565a-5ada-4575-bda0-c951197bc6b1-0-00001.parquet      |PARQUET    |1           |1696              |{2026-09-10}|
|s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-09-12/00000-3-480f326e-83c0-4f1d-bc55-77391e5685b4-0-00003.parquet       |PARQUET    |1           |1662              |{2026-09-12}|
|s3a://warehouse/eco

In [19]:
%%sql
-- 1. Initial Ingestion
INSERT INTO demo.ecommerce.orders VALUES
    (1101, 5501, CAST('2026-09-10 08:15:00' AS TIMESTAMP), 'PENDING', 120.50),
    (1102, 5502, CAST('2026-09-11 09:30:00' AS TIMESTAMP), 'PENDING', 89.99),
    (1103, 5503, CAST('2026-09-12 14:45:00' AS TIMESTAMP), 'PROCESSING', 1045.00);

-- 2. Row-level Update
UPDATE demo.ecommerce.orders 
SET status = 'PROCESSING' 
WHERE order_id = 1001;

-- 3. Row-level Delete
DELETE FROM demo.ecommerce.orders 
WHERE order_id = 1002;

In [20]:
%%sql
-- 3. Expire historical snapshots (Retaining the last 2 to save storage)
CALL demo.system.expire_snapshots(
  table => 'ecommerce.orders', 
  retain_last => 1
);

-- 4. Sweep orphan physical files that failed to commit due to job crashes
CALL demo.system.remove_orphan_files(
  table => 'ecommerce.orders'
);

+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|0                       |0                                  |0                                  |0                           |0                           |0                             |
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+

+--------------------+
|orphan_file_location|
+------------

***Why Returned Zero***
    
1. Why expire_snapshots returned zero

In a production environment, if Iceberg immediately deleted old data files the second a new snapshot was created, any long-running reporting queries or machine learning jobs currently reading that older snapshot would suddenly crash with a "File Not Found" error.

To prevent this, Iceberg has a built-in Time-Based Safety Lock.
Even if you explicitly set retain_last => 1, Iceberg will silently ignore that command if the snapshots are too new. By default, Iceberg will not expire any snapshot created within the last 48 hours (or 5 days, depending on engine defaults). Because you just ran all these inserts and updates a few minutes ago, Iceberg is protecting them.

Because no snapshots were expired, no physical data files, manifest files, or delete files were removed. Hence, the deleted_data_files_count is 0.

2. Why remove_orphan_files returned zero
An "orphan file" is a physical Parquet file sitting in MinIO that is completely detached from the Iceberg metadata tree (usually caused by a Spark job crashing mid-write).

Your Spark jobs completed perfectly, so no orphans were created.

Because expire_snapshots didn't run, no historical files were detached from the metadata.

Safety Lock: Just like expiration, remove_orphan_files has a default 3-day grace period. It assumes any "untracked" file created in the last 3 days might just belong to a slow-running Spark job that hasn't finished committing yet.

How to override the safety locks (For Testing)
If you are doing local testing and want to physically force Iceberg to delete the history and the files right now, you have to explicitly override the time-based safety lock by passing the older_than argument with the current timestamp.

Try running this block. You will see the numbers jump up as Iceberg forcefully shreds the historical files from MinIO:

In [22]:
%%sql
-- Force expire everything except the latest snapshot using a literal timestamp
CALL demo.system.expire_snapshots(
  table => 'ecommerce.orders', 
  retain_last => 1,
  older_than => TIMESTAMP '2026-09-13 23:30:00'
);

-- Force sweep orphans immediately using a literal timestamp
CALL demo.system.remove_orphan_files(
  table => 'ecommerce.orders',
  older_than => TIMESTAMP '2026-09-13 23:30:00'
);

+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|1                       |0                                  |0                                  |5                           |8                           |0                             |
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+



IllegalArgumentException: Cannot remove orphan files with an interval less than 24 hours. Executing this procedure with a short interval may corrupt the table if other operations are happening at the same time. If you are absolutely confident that no concurrent operations will be affected by removing orphan files with such a short interval, you can use the Action API to remove orphan files with an arbitrary interval.

This is another excellent architectural lesson in Apache Iceberg!

First, notice the first table in your output: expire_snapshots worked perfectly! By using the literal timestamp, you successfully forced Iceberg to delete 1 physical Parquet data file, 5 manifest files, and 8 manifest lists that were no longer needed by the latest snapshot.

However, the second command (remove_orphan_files) hit a hardcoded safety wall.

Why did remove_orphan_files fail?
The error message says it all: Cannot remove orphan files with an interval less than 24 hours.

While expire_snapshots lets you recklessly delete history up to the current minute, remove_orphan_files has a strict, non-bypassable (via SQL) minimum buffer of 24 hours.

Why? Because an "orphan" file is just a file in your S3/MinIO bucket that Iceberg doesn't recognize yet. If a massive Spark job is currently in the middle of a 2-hour write operation, those files are technically "orphans" until the final commit happens. If you sweep orphans with an interval of a few minutes, you will delete the files being written by active jobs, immediately corrupting them.

The Fix
To execute this successfully in your notebook without an error, your older_than timestamp for remove_orphan_files must be at least 24 hours in the past.

Here is the corrected execution block. (Notice we set the orphan sweep to exactly 24 hours prior).

SQL
%%sql
-- 1. Expire snapshots works with the recent timestamp
CALL demo.system.expire_snapshots(
  table => 'ecommerce.orders', 
  retain_last => 1,
  older_than => TIMESTAMP '2026-09-13 23:30:00'
);

-- 2. Sweep orphans requires a minimum 24-hour buffer!
-- We use yesterday's date to satisfy the safety check.
CALL demo.system.remove_orphan_files(
  table => 'ecommerce.orders',
  older_than => TIMESTAMP '2026-09-12 23:30:00'
);
When you run this, the orphan files procedure will execute smoothly. It will return a table with orphan_file_location showing 0 rows, accurately proving your bucket is clean and no active writes were accidentally destroyed!

In [16]:
%%sql
-- Files (Physical Parquet data and positional delete files)
SELECT file_path, file_format, record_count, file_size_in_bytes, partition 
FROM demo.ecommerce.orders.files;

+----------------------------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+------------+
|file_path                                                                                                                               |file_format|record_count|file_size_in_bytes|partition   |
+----------------------------------------------------------------------------------------------------------------------------------------+-----------+------------+------------------+------------+
|s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-09-10/00000-23-9893565a-5ada-4575-bda0-c951197bc6b1-0-00001.parquet      |PARQUET    |1           |1696              |{2026-09-10}|
|s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-09-12/00000-3-480f326e-83c0-4f1d-bc55-77391e5685b4-0-00003.parquet       |PARQUET    |1           |1662              |{2026-09-12}|
|s3a://warehouse/eco